# 5. XGBoost forecasts

This notebook forecasts the next 4 weeks of demand for Levi's and The Souled Store with XGBoost.

It follows notebook 4. The weekly files, Monday week labels, lag features, 20-week history rule, and recent-average fallback are the same. The future forecast is recursive: week 2 uses the week 1 prediction, week 3 uses the week 1 and week 2 predictions, and week 4 uses the three predictions already made. Unknown future sales are not used as inputs.

What this notebook does:

1. Load the same four processed weekly datasets.
2. Put every dataset on the same Monday week label.
3. Build the same lag features from earlier observations only.
4. Run a small sanity test, including a check that the eligible series and cutoff dates match the saved Random Forest evaluation.
5. Score XGBoost on those same series, cutoffs, and 4-week horizons.
6. Compare XGBoost with the saved Random Forest results and a repeat-last-week baseline.
7. Forecast 4 future weeks for every location-item series and save new files.

Notebooks 1 to 4 are left as they are. Raw and processed data are only read. Existing output files are not overwritten.

Run the cells from top to bottom. The scoring cell trains one small model per eligible series and cutoff. A sanity timing print appears before that cell.


## 1. Import libraries


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from xgboost import XGBRegressor

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

timings = {}


## 2. Load the four weekly datasets

These are the same processed files used in notebook 4. This notebook reads them and does not change the raw or processed data.


In [ ]:
levis_sku = pd.read_csv(
    "../data/processed/levis_Order_data_cleaned_weekly_filled.csv"
)
souledstore_sku = pd.read_csv(
    "../data/processed/thesouledstore_Oder_data_cleaned_weekly_filled.csv"
)
levis_category = pd.read_csv(
    "../data/processed/levis_category_weekly.csv"
)
souledstore_category = pd.read_csv(
    "../data/processed/souledstore_category_weekly.csv"
)

datasets = {
    "Levis SKU": levis_sku,
    "Souled Store SKU": souledstore_sku,
    "Levis Category": levis_category,
    "Souled Store Category": souledstore_category,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))


## 3. Check dates, missing values, and week labels

The SKU files label each week with a Monday. The category files label the same week with the following Tuesday. Demand values are checked here before any week label is moved.


In [ ]:
for name, df in datasets.items():
    df["Week"] = pd.to_datetime(df["Week"], errors="coerce")
    weekdays = sorted(df["Week"].dt.day_name().unique())

    print(f"\n--- {name} ---")
    print("Date range:", df["Week"].min().date(), "to", df["Week"].max().date())
    print("Weekday labels:", weekdays)
    print("Missing values:\n", df.isnull().sum())
    print("Negative demand rows:", int((df["Net Sold Quantity"] < 0).sum()))
    print("Duplicate rows:", int(df.duplicated().sum()))


## 4. Use one Monday label for every dataset

A Tuesday category date is the day after the Monday used in the SKU files. For example, category week `2026-02-24` is the same week as SKU week `2026-02-23`.

The label is moved back by one day. `Net Sold Quantity` is not changed, and no rows are added or removed. Running this cell again is safe: once the weeks are already Mondays, they are left as they are.


In [ ]:
def prepare_weekly(df, item_col):
    """Sort one series at a time and align Tuesday labels to Monday."""
    out = df.copy()
    out["Week"] = pd.to_datetime(out["Week"])
    weekdays = set(out["Week"].dt.day_name().unique())

    if weekdays == {"Tuesday"}:
        out["Week"] = out["Week"] - pd.Timedelta(days=1)
        shifted = True
    elif weekdays == {"Monday"}:
        shifted = False
    else:
        raise ValueError(
            "Expected every week to fall on Monday or Tuesday. "
            f"Found {sorted(weekdays)}."
        )

    out = out.sort_values(["Fulfilment Location", item_col, "Week"])
    key = ["Fulfilment Location", item_col, "Week"]
    if out.duplicated(key).any():
        raise ValueError("Duplicate location-item-week rows after week alignment.")

    if set(out["Week"].dt.day_name().unique()) != {"Monday"}:
        raise ValueError("Week alignment did not produce Monday labels.")

    return out.reset_index(drop=True), shifted


demand_before = {
    "Levis Category": float(levis_category["Net Sold Quantity"].sum()),
    "Souled Store Category": float(souledstore_category["Net Sold Quantity"].sum()),
}

levis_sku, levis_sku_shifted = prepare_weekly(levis_sku, "SKU Code")
souledstore_sku, souled_sku_shifted = prepare_weekly(souledstore_sku, "SKU Code")
levis_category, levis_category_shifted = prepare_weekly(levis_category, "Category")
souledstore_category, souled_category_shifted = prepare_weekly(
    souledstore_category, "Category1"
)

datasets = {
    "Levis SKU": (levis_sku, "SKU Code", levis_sku_shifted),
    "Souled Store SKU": (souledstore_sku, "SKU Code", souled_sku_shifted),
    "Levis Category": (levis_category, "Category", levis_category_shifted),
    "Souled Store Category": (
        souledstore_category,
        "Category1",
        souled_category_shifted,
    ),
}

for name, (df, item_col, shifted) in datasets.items():
    print(f"\n--- {name} ---")
    print("Tuesday labels moved to the previous Monday:", shifted)
    print("Date range:", df["Week"].min().date(), "to", df["Week"].max().date())
    print("Locations:", df["Fulfilment Location"].nunique())
    print("Series:", df.groupby(["Fulfilment Location", item_col]).ngroups)

print("\nCategory demand sum, before -> after")
for name in ["Levis Category", "Souled Store Category"]:
    after = float(datasets[name][0]["Net Sold Quantity"].sum())
    print(f"{name}: {demand_before[name]} -> {after}")
    if demand_before[name] != after:
        raise ValueError("Week alignment changed demand values.")


## 5. Create lag features

The model predicts a week from demand that was already known before that week:

- `lag_1` is the previous stored week in that series
- `lag_2` is two stored weeks earlier
- `lag_4` is four stored weeks earlier
- `rolling_mean_4` is the average of those previous four stored weeks

Each calculation stays inside one location-item series, after the series is sorted by week. The week being predicted is not one of its own inputs.

In the filled SKU files, stored weeks are consecutive Mondays. A category series can skip a week. This notebook does not invent a row for a missing week, and it does not fill that gap with zero. The previous stored observation is used, which is the same rule as notebook 4.

The cell below also checks that the feature table and the recursive-forecast helper describe the same lags.


In [ ]:
LAG_FEATURES = ["lag_1", "lag_2", "lag_4", "rolling_mean_4"]


def create_features(df, item_col):
    df = df.copy()
    df = df.sort_values(["Fulfilment Location", item_col, "Week"])

    group = df.groupby(["Fulfilment Location", item_col])["Net Sold Quantity"]
    df["lag_1"] = group.shift(1)
    df["lag_2"] = group.shift(2)
    df["lag_4"] = group.shift(4)
    df["rolling_mean_4"] = group.transform(
        lambda s: s.shift(1).rolling(4).mean()
    )

    df = df.dropna(subset=LAG_FEATURES)
    return df


def lag_row_from_history(history):
    """Features for the next week, using only the history so far."""
    if len(history) < 4:
        raise ValueError("Lag features need 4 earlier weeks.")
    return {
        "lag_1": history[-1],
        "lag_2": history[-2],
        "lag_4": history[-4],
        "rolling_mean_4": float(np.mean(history[-4:])),
    }


def assert_feature_definitions_match(df, item_col):
    """The table of lags and the recursive helper must agree."""
    counts = df.groupby(["Fulfilment Location", item_col]).size()
    location, item = counts.idxmax()
    series = df[
        (df["Fulfilment Location"] == location) & (df[item_col] == item)
    ].sort_values("Week")
    history = series["Net Sold Quantity"].astype(float).tolist()
    featured = create_features(series, item_col).sort_values("Week")
    if len(featured) != len(history) - 4:
        raise AssertionError("Feature rows do not equal history length minus 4.")
    for offset, row in enumerate(featured.itertuples(index=False)):
        built = lag_row_from_history(history[: offset + 4])
        for name in LAG_FEATURES:
            if not np.isclose(getattr(row, name), built[name]):
                raise AssertionError(f"{name} does not match the history helper.")
    return location, item, len(history)


for name, (df, item_col, _) in datasets.items():
    location, item, weeks = assert_feature_definitions_match(df, item_col)
    print(f"{name}: lag definitions match on {location} / {item} ({weeks} weeks)")

example_history = [10, 0, 4, 6, 8]
example_row = lag_row_from_history(example_history)
print("\nExample history:", example_history)
print("Features for the following week:", example_row)
if example_row["lag_1"] != 8 or example_row["lag_4"] != 0:
    raise AssertionError("The example lag row is not the previous stored demand.")
if not np.isclose(example_row["rolling_mean_4"], np.mean([0, 4, 6, 8])):
    raise AssertionError("The 4-week average includes the wrong weeks.")


## 6. Short histories and the XGBoost model

A series needs at least 20 observed weeks before XGBoost is trained. The first 4 weeks only create lags, so 20 weeks leave 16 weeks for the model to learn from. A shorter series does not get invented history. Its forecast is the average of the available recent demand: the last 4 weeks when those weeks exist, otherwise every week the series has. That same average is used for all 4 future weeks and labeled `recent_average_fallback`. This is the fallback from notebook 4.

XGBoost settings, chosen before looking at the scores and not tuned on the holdout:

- 100 trees, the same number used by the Random Forest in notebook 4
- maximum depth 4, the same depth limit
- `min_child_weight=5`, so a leaf must cover at least 5 training rows under squared error. This plays the same role as `min_samples_leaf=5` in notebook 4
- learning rate 0.1, which is the extra setting boosting needs. Each new tree is allowed to move the prediction by a small step
- `random_state=42`, so a repeated run uses the same tree construction

Each model sees only one series. Predictions are kept at zero or above because sold quantity cannot be negative.


In [ ]:
MIN_HISTORY_WEEKS = 20
N_ROLLING_ORIGINS = 8
EXPECTED_ORIGINS = [
    "2025-12-08",
    "2025-12-15",
    "2025-12-22",
    "2025-12-29",
    "2026-01-05",
    "2026-01-12",
    "2026-01-19",
    "2026-01-26",
]
EXPECTED_PAIRS = {
    "Levis SKU": 224,
    "Souled Store SKU": 1461,
    "Levis Category": 15,
    "Souled Store Category": 528,
}
XGB_PARAMS = dict(
    n_estimators=100,
    max_depth=4,
    min_child_weight=5,
    learning_rate=0.1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    verbosity=0,
)


def recent_average(history):
    """Average of the latest available demand, up to 4 weeks."""
    recent = history[-4:] if len(history) >= 4 else history
    return float(np.mean(recent))


def fit_xgboost(history):
    """Train on one series. Columns are lag_1, lag_2, lag_4, rolling_mean_4."""
    rows = []
    targets = []
    for t in range(4, len(history)):
        rows.append(lag_row_from_history(history[:t]))
        targets.append(history[t])
    if len(rows) < 1:
        raise ValueError("XGBoost needs at least one training week.")
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(pd.DataFrame(rows)[LAG_FEATURES], targets)
    return model


def recursive_xgboost_forecast(history):
    """Forecast 4 weeks. Later weeks use earlier predictions."""
    model = fit_xgboost(history)
    working = list(history)
    predictions = []
    for _ in range(4):
        row = pd.DataFrame([lag_row_from_history(working)])
        prediction = max(0.0, float(model.predict(row[LAG_FEATURES])[0]))
        predictions.append(prediction)
        working.append(prediction)
    return predictions


## 7. Sanity test

This cell does three cheap checks before the full scoring run.

1. One real series is trained only through Monday 2026-01-26. Later actual weeks exist, and they are held out of the training history. The second forecast step must use the first prediction as `lag_1`.
2. Eight models are timed, so the full run can be judged before it starts.
3. Repeat-last-week error is recomputed for every dataset with no XGBoost training. The cutoff dates, series-cutoff counts, and baseline errors must match `outputs/ml_rolling_origin_evaluation.csv`. That file is the Random Forest comparison from notebook 4. Random Forest is not trained again here.


In [ ]:
output_dir = Path("../outputs")
published = pd.read_csv(output_dir / "ml_rolling_origin_evaluation.csv")


def rolling_origin_evaluation(df, item_col, train_xgboost):
    """Score 4-week forecasts at the same past cutoffs used in notebook 4."""
    weeks = [pd.Timestamp(week) for week in sorted(df["Week"].unique())]
    origins = weeks[:-4][-N_ROLLING_ORIGINS:]
    scored_rows = []
    scored_series = set()
    total_series = 0
    short_series = 0
    long_unscored = 0
    skip_short = 0
    skip_missing_origin = 0
    skip_missing_future = 0

    for key, group in df.groupby(["Fulfilment Location", item_col], sort=False):
        total_series += 1
        group = group.sort_values("Week")
        week_list = [pd.Timestamp(week) for week in group["Week"]]
        values = group["Net Sold Quantity"].astype(float).tolist()
        position = {week: index for index, week in enumerate(week_list)}
        if len(values) < MIN_HISTORY_WEEKS:
            short_series += 1
        series_scored = False

        for origin in origins:
            pos = position.get(origin)
            if pos is None:
                skip_missing_origin += 1
                continue
            if pos + 1 < MIN_HISTORY_WEEKS:
                skip_short += 1
                continue
            future_weeks = [
                origin + pd.Timedelta(days=7 * step) for step in (1, 2, 3, 4)
            ]
            if any(week not in position for week in future_weeks):
                skip_missing_future += 1
                continue

            history = values[: pos + 1]
            if len(history) < MIN_HISTORY_WEEKS:
                raise AssertionError("Eligible history is shorter than 20 weeks.")
            if max(week_list[: pos + 1]) > origin:
                raise AssertionError("Training history goes past the cutoff.")

            predictions = (
                recursive_xgboost_forecast(history)
                if train_xgboost
                else [np.nan, np.nan, np.nan, np.nan]
            )
            baseline = history[-1]
            for horizon, (prediction, week) in enumerate(
                zip(predictions, future_weeks), start=1
            ):
                if week <= origin:
                    raise AssertionError("A scored week was not after the cutoff.")
                actual = values[position[week]]
                scored_rows.append(
                    {
                        "horizon": horizon,
                        "xgb_error": abs(float(prediction) - actual),
                        "baseline_error": abs(baseline - actual),
                    }
                )
            series_scored = True

        if series_scored:
            scored_series.add(key)
        elif len(values) >= MIN_HISTORY_WEEKS:
            long_unscored += 1

    counts = {
        "total_series": total_series,
        "series_scored": len(scored_series),
        "short_series": short_series,
        "long_unscored": long_unscored,
        "pairs": len(scored_rows) // 4,
        "skip_short": skip_short,
        "skip_missing_origin": skip_missing_origin,
        "skip_missing_future": skip_missing_future,
    }
    if counts["series_scored"] + short_series + long_unscored != total_series:
        raise AssertionError("Series counts do not add up.")
    return origins, pd.DataFrame(scored_rows), counts


def assert_same_evaluation_design(dataset_name, origins, scores):
    """Stop if this run would score a different set from notebook 4."""
    origin_labels = [origin.date().isoformat() for origin in origins]
    if origin_labels != EXPECTED_ORIGINS:
        raise AssertionError(f"{dataset_name} cutoffs are {origin_labels}.")
    published_part = published[published["Dataset"] == dataset_name]
    if published_part.empty:
        raise AssertionError(f"{dataset_name} is missing from the saved evaluation.")

    for horizon in (1, 2, 3, 4):
        part = scores[scores["horizon"] == horizon]
        label = f"{horizon} week ahead"
        row = published_part[published_part["Horizon"] == label]
        if len(row) != 1:
            raise AssertionError(f"Missing published row for {dataset_name} {label}.")
        row = row.iloc[0]
        got_mae = round(float(part["baseline_error"].mean()), 2)
        got_median = round(float(part["baseline_error"].median()), 2)
        if len(part) != int(row["Scored weeks"]):
            raise AssertionError(f"{dataset_name} {label} scored a different number of weeks.")
        if got_mae != round(float(row["Repeat last week MAE"]), 2):
            raise AssertionError(
                f"{dataset_name} {label} baseline MAE {got_mae} "
                f"does not match {row['Repeat last week MAE']}."
            )
        if got_median != round(float(row["Repeat last week median AE"]), 2):
            raise AssertionError(
                f"{dataset_name} {label} baseline median {got_median} "
                f"does not match {row['Repeat last week median AE']}."
            )

    overall = published_part[published_part["Horizon"] == "All 4 weeks"].iloc[0]
    got_mae = round(float(scores["baseline_error"].mean()), 2)
    got_median = round(float(scores["baseline_error"].median()), 2)
    if len(scores) != int(overall["Scored weeks"]):
        raise AssertionError(f"{dataset_name} all-horizon week count does not match.")
    if got_mae != round(float(overall["Repeat last week MAE"]), 2):
        raise AssertionError(f"{dataset_name} all-horizon baseline MAE does not match.")
    if got_median != round(float(overall["Repeat last week median AE"]), 2):
        raise AssertionError(f"{dataset_name} all-horizon baseline median does not match.")
    if len(scores) // 4 != EXPECTED_PAIRS[dataset_name]:
        raise AssertionError(
            f"{dataset_name} pair count {len(scores) // 4} "
            f"does not match {EXPECTED_PAIRS[dataset_name]}."
        )


cutoff = pd.Timestamp("2026-01-26")
sanity_name = "Souled Store Category"
sanity_df, sanity_item, _ = datasets[sanity_name]
sanity_key = None
sanity_group = None
for key, group in sanity_df.groupby(["Fulfilment Location", sanity_item], sort=False):
    group = group.sort_values("Week")
    through_cutoff = group[group["Week"] <= cutoff]
    after_cutoff = group[group["Week"] > cutoff]
    if len(through_cutoff) >= MIN_HISTORY_WEEKS and len(after_cutoff) >= 4:
        if through_cutoff["Week"].max() == cutoff:
            sanity_key = key
            sanity_group = group
            break
if sanity_group is None:
    raise AssertionError("Could not find a series for the sanity test.")

train_frame = sanity_group[sanity_group["Week"] <= cutoff]
holdout_frame = sanity_group[sanity_group["Week"] > cutoff]
history = train_frame["Net Sold Quantity"].astype(float).tolist()
if train_frame["Week"].max() != cutoff or holdout_frame["Week"].min() <= cutoff:
    raise AssertionError("The sanity split mixed future weeks into training.")

started = time.perf_counter()
model = fit_xgboost(history)
fit_seconds = time.perf_counter() - started
working = list(history)
first_features = lag_row_from_history(working)
first = max(0.0, float(model.predict(pd.DataFrame([first_features])[LAG_FEATURES])[0]))
working.append(first)
second_features = lag_row_from_history(working)
if second_features["lag_1"] != first:
    raise AssertionError("Week 2 did not use the week 1 prediction.")
if min(first, second_features["lag_1"]) < 0:
    raise AssertionError("A sanity prediction was negative.")

print("Sanity series:", sanity_key[0], "/", sanity_key[1])
print("Training weeks through", cutoff.date(), ":", len(history))
print("Held-out actual weeks after the cutoff:", len(holdout_frame))
print("Training rows after the 4-week lag warmup:", len(history) - 4)
print("First-step features:", {name: round(first_features[name], 2) for name in LAG_FEATURES})
print("First prediction:", round(first, 2))
print("Second-step lag_1 equals that prediction:", round(second_features["lag_1"], 2))
print(f"One fit took {fit_seconds:.3f} seconds.")

longest = (
    sanity_df.groupby(["Fulfilment Location", sanity_item]).size().sort_values(ascending=False).head(8)
)
started = time.perf_counter()
for location, item in longest.index:
    group = sanity_df[
        (sanity_df["Fulfilment Location"] == location) & (sanity_df[sanity_item] == item)
    ].sort_values("Week")
    sample_history = group["Net Sold Quantity"].astype(float).tolist()
    recursive_xgboost_forecast(sample_history)
eight_fit_seconds = time.perf_counter() - started
print(f"Eight recursive forecasts took {eight_fit_seconds:.2f} seconds.")
timings["sanity_eight_forecasts_seconds"] = round(eight_fit_seconds, 2)

print("\nChecking the eligible series against the saved Random Forest evaluation...")
started = time.perf_counter()
for name, (df, item_col, _) in datasets.items():
    origins, scores, counts = rolling_origin_evaluation(df, item_col, train_xgboost=False)
    assert_same_evaluation_design(name, origins, scores)
    print(
        f"{name}: {counts['pairs']} series-cutoff pairs, "
        f"{counts['series_scored']} series scored, "
        f"{counts['short_series']} series shorter than 20 weeks, "
        f"{counts['long_unscored']} long series left unscored."
    )
design_seconds = time.perf_counter() - started
timings["design_check_seconds"] = round(design_seconds, 2)
print(f"Design check took {design_seconds:.1f} seconds.")
print("Sanity test passed. The full XGBoost scoring cell is next.")


## 8. Score the 4-week recursive forecast

There are 8 cutoffs, one week apart, from 2025-12-08 through 2026-01-26. After the last cutoff, the next four observed weeks run through 2026-02-23.

A series is scored at a cutoff only when three things are true:

- that cutoff week was observed
- the series already has at least 20 stored weeks through the cutoff
- the next four calendar weeks were also observed for that series

Actual demand after the cutoff is used only to score the forecast. XGBoost is trained on weeks up to and including the cutoff, predicts the next week, and then feeds that prediction into the following week.

The baseline repeats the cutoff week's demand for all four weeks. Random Forest errors are read from the saved rolling-origin file after the sanity cell has shown that the baseline and the series counts match. Those Random Forest models are not fit again.

MAE is the average absolute error. A few very large weeks can make this number big. Median AE is the error of a typical week. Both are reported. A method can be closer on the average and further away on a typical week.


In [ ]:
def lowest_method(scores_by_method):
    """Name the method or methods with the lowest rounded score."""
    rounded = {
        name: round(float(value), 2) for name, value in scores_by_method.items()
    }
    best = min(rounded.values())
    winners = [name for name, value in rounded.items() if value == best]
    if len(winners) == 1:
        return winners[0]
    return "Tie: " + " and ".join(winners)


def evaluation_rows(dataset_name, origins, scores, counts):
    cutoff_start = origins[0].date().isoformat()
    cutoff_end = origins[-1].date().isoformat()
    published_part = published[published["Dataset"] == dataset_name]
    rows = []

    def one_row(label, part):
        published_row = published_part[published_part["Horizon"] == label].iloc[0]
        xgb_mae = float(part["xgb_error"].mean())
        baseline_mae = float(part["baseline_error"].mean())
        forest_mae = float(published_row["Random Forest MAE"])
        xgb_median = float(part["xgb_error"].median())
        baseline_median = float(part["baseline_error"].median())
        forest_median = float(published_row["Random Forest median AE"])
        return {
            "Dataset": dataset_name,
            "Horizon": label,
            "Cutoff start": cutoff_start,
            "Cutoff end": cutoff_end,
            "Total series": counts["total_series"],
            "Series scored": counts["series_scored"],
            "Series excluded": counts["short_series"] + counts["long_unscored"],
            "Excluded because history shorter than 20 weeks": counts["short_series"],
            "Excluded because a long history could not be scored": counts["long_unscored"],
            "Series-origin pairs": counts["pairs"],
            "Scored weeks": len(part),
            "Cutoff pairs skipped because history was still short": counts["skip_short"],
            "Cutoff pairs skipped because the cutoff week was absent": counts[
                "skip_missing_origin"
            ],
            "Cutoff pairs skipped because a future week was absent": counts[
                "skip_missing_future"
            ],
            "XGBoost MAE": round(xgb_mae, 2),
            "XGBoost median AE": round(xgb_median, 2),
            "Random Forest MAE": round(forest_mae, 2),
            "Random Forest median AE": round(forest_median, 2),
            "Repeat last week MAE": round(baseline_mae, 2),
            "Repeat last week median AE": round(baseline_median, 2),
            "XGBoost closer than repeat last week": int(
                (part["xgb_error"] < part["baseline_error"]).sum()
            ),
            "Repeat last week closer than XGBoost": int(
                (part["baseline_error"] < part["xgb_error"]).sum()
            ),
            "Ties": int((part["xgb_error"] == part["baseline_error"]).sum()),
            "Lowest MAE": lowest_method(
                {
                    "XGBoost": xgb_mae,
                    "Random Forest": forest_mae,
                    "Repeat last week": baseline_mae,
                }
            ),
            "Lowest median AE": lowest_method(
                {
                    "XGBoost": xgb_median,
                    "Random Forest": forest_median,
                    "Repeat last week": baseline_median,
                }
            ),
        }

    for horizon in (1, 2, 3, 4):
        rows.append(one_row(f"{horizon} week ahead", scores[scores["horizon"] == horizon]))
    rows.append(one_row("All 4 weeks", scores))
    return rows


started = time.perf_counter()
evaluation_tables = []
evaluation_details = {}
for name, (df, item_col, _) in datasets.items():
    print(f"Scoring {name}...")
    origins, scores, counts = rolling_origin_evaluation(
        df, item_col, train_xgboost=True
    )
    if scores["xgb_error"].isna().any():
        raise AssertionError(f"{name} has a missing XGBoost error.")
    assert_same_evaluation_design(name, origins, scores)
    table = pd.DataFrame(evaluation_rows(name, origins, scores, counts))
    evaluation_tables.append(table)
    evaluation_details[name] = counts
    overall = table[table["Horizon"] == "All 4 weeks"].iloc[0]
    print(
        f"  MAE over all 4 weeks: XGBoost {overall['XGBoost MAE']:.2f}, "
        f"Random Forest {overall['Random Forest MAE']:.2f}, "
        f"repeat last week {overall['Repeat last week MAE']:.2f}."
    )

evaluation_df = pd.concat(evaluation_tables, ignore_index=True)
evaluation_seconds = time.perf_counter() - started
timings["xgboost_evaluation_seconds"] = round(evaluation_seconds, 2)
print(f"\nXGBoost evaluation took {evaluation_seconds:.1f} seconds.")
print(evaluation_df.to_string(index=False))


## 9. Save the XGBoost evaluation

The file records MAE and median absolute error by dataset and horizon, with the series counts and exclusions. Random Forest columns are the saved notebook 4 results for this same design. This cell does not replace `ml_model_evaluation.csv` or `ml_rolling_origin_evaluation.csv`.


In [ ]:
PROTECTED_OUTPUTS = {
    "ml_model_evaluation.csv",
    "ml_rolling_origin_evaluation.csv",
    "forecast_history_freshness.csv",
    "levis_sku_4_week_forecast.csv",
    "levis_category_4_week_forecast.csv",
    "souledstore_sku_4_week_forecast.csv",
    "souledstore_category_4_week_forecast.csv",
    "levis_sku_ml_forecast.csv",
    "souledstore_sku_ml_forecast.csv",
    "levis_category_ml_forecast.csv",
    "souledstore_category_ml_forecast.csv",
    "levis_sku_model_evaluation.csv",
    "souledstore_category_model_evaluation.csv",
}


def save_output(filename, frame):
    if filename in PROTECTED_OUTPUTS:
        raise AssertionError(f"Refusing to overwrite {filename}.")
    path = output_dir / filename
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame)} rows to {path.resolve()}")
    return path


save_output("xgboost_model_evaluation.csv", evaluation_df)


## 10. Forecast the next 4 weeks

Every location-item combination gets four future weeks. The four weeks are the same calendar dates for every series: the next four Mondays after the dataset's last week.

A series with at least 20 observed weeks gets its own XGBoost model, trained on all of that series' history. The model then forecasts one week at a time and appends each prediction before forecasting the next week.

A shorter series uses `recent_average_fallback`. Missing weeks are not filled with zeros or made-up demand.

Some series end before February 2026. They still receive the common 4 future weeks, using their own last observations. `Last Observed Week` shows when that history stopped.

These forecast models are fit again on all available history. They are not the cutoff models from the evaluation above. The evaluation answers how accurate the method was on known weeks. The forecast files answer what the method predicts after the last observed week.


In [ ]:
def forecast_next_4_weeks(df, item_col):
    df = df.sort_values(["Fulfilment Location", item_col, "Week"]).copy()
    last_week = df["Week"].max()
    future_weeks = pd.date_range(
        last_week + pd.Timedelta(days=7),
        periods=4,
        freq="7D",
    )

    forecasts = []
    grouped = df.groupby(["Fulfilment Location", item_col], sort=True)
    for (location, item), group in grouped:
        group = group.sort_values("Week")
        history = group["Net Sold Quantity"].astype(float).tolist()
        if len(history) == 0:
            continue

        last_observed = group["Week"].max()
        if len(history) >= MIN_HISTORY_WEEKS:
            method = "xgboost"
            predictions = recursive_xgboost_forecast(history)
        else:
            method = "recent_average_fallback"
            predictions = [recent_average(history)] * 4

        for week, prediction in zip(future_weeks, predictions):
            forecasts.append(
                {
                    "Week": week,
                    "Fulfilment Location": location,
                    item_col: item,
                    "Forecast Net Sold Quantity": round(prediction, 2),
                    "Forecast Method": method,
                    "History Weeks": len(history),
                    "Last Observed Week": last_observed,
                }
            )

    forecast = pd.DataFrame(forecasts)
    return forecast.sort_values(
        ["Fulfilment Location", item_col, "Week"]
    ).reset_index(drop=True)


forecast_inputs = [
    ("Levis SKU", levis_sku, "SKU Code"),
    ("Souled Store SKU", souledstore_sku, "SKU Code"),
    ("Levis Category", levis_category, "Category"),
    ("Souled Store Category", souledstore_category, "Category1"),
]

started = time.perf_counter()
forecast_results = {}
for name, df, item_col in forecast_inputs:
    print(f"Forecasting {name}...")
    forecast = forecast_next_4_weeks(df, item_col)
    forecast_results[name] = forecast
    methods = forecast.groupby("Forecast Method")["History Weeks"].size() / 4
    print(forecast["Week"].dt.date.unique().tolist())
    print("Series by method:")
    print(methods.astype(int).to_string())
    print(f"Rows: {len(forecast)}")

forecast_seconds = time.perf_counter() - started
timings["forecast_seconds"] = round(forecast_seconds, 2)
print(f"\nForecasting took {forecast_seconds:.1f} seconds.")


## 11. Save the four XGBoost forecast files


In [ ]:
file_names = {
    "Levis SKU": "levis_sku_xgboost_forecast.csv",
    "Souled Store SKU": "souledstore_sku_xgboost_forecast.csv",
    "Levis Category": "levis_category_xgboost_forecast.csv",
    "Souled Store Category": "souledstore_category_xgboost_forecast.csv",
}

for name, forecast in forecast_results.items():
    saved = forecast.copy()
    saved["Week"] = pd.to_datetime(saved["Week"]).dt.strftime("%Y-%m-%d")
    saved["Last Observed Week"] = pd.to_datetime(
        saved["Last Observed Week"]
    ).dt.strftime("%Y-%m-%d")
    save_output(file_names[name], saved)


## 12. Validate the saved forecasts

This cell reads the new CSVs back from `outputs/` and checks the files that were just written.

Each series must have exactly four future weeks, with no duplicate location-item-week rows and no negative forecasts. XGBoost is allowed only when the series has at least 20 weeks. The fallback is allowed only for shorter series, and its four weeks must share one number.

`Last Observed Week` shows the last real week for that series. A history is recent when that week is 2026-01-26 or later, which is the four weeks leading up to the dataset's final week of 2026-02-23. A history is stale when the last observation is older than that. Stale series still receive a March forecast, but it is based on older demand. The existing freshness file from notebook 4 is left unchanged.


In [ ]:
expected_weeks = ["2026-03-02", "2026-03-09", "2026-03-16", "2026-03-23"]
original_data = {
    "Levis SKU": (levis_sku, "SKU Code"),
    "Souled Store SKU": (souledstore_sku, "SKU Code"),
    "Levis Category": (levis_category, "Category"),
    "Souled Store Category": (souledstore_category, "Category1"),
}

for name, filename in file_names.items():
    original, item_col = original_data[name]
    forecast = pd.read_csv(output_dir / filename)
    forecast["Week"] = pd.to_datetime(forecast["Week"])
    forecast["Last Observed Week"] = pd.to_datetime(forecast["Last Observed Week"])

    print(f"\n--- {name} ---")
    keys = ["Week", "Fulfilment Location", item_col]
    combo_counts = forecast.groupby(["Fulfilment Location", item_col]).size()
    original_combos = set(
        map(tuple, original[["Fulfilment Location", item_col]].drop_duplicates().to_numpy())
    )
    forecast_combos = set(
        map(tuple, forecast[["Fulfilment Location", item_col]].drop_duplicates().to_numpy())
    )
    weeks = sorted(forecast["Week"].dt.strftime("%Y-%m-%d").unique())
    duplicate_keys = int(forecast.duplicated(keys).sum())
    negative = int((forecast["Forecast Net Sold Quantity"] < 0).sum())
    missing = int(forecast["Forecast Net Sold Quantity"].isna().sum())

    print("Rows:", len(forecast))
    print("Series:", forecast.groupby(["Fulfilment Location", item_col]).ngroups)
    print("Weeks:", weeks)
    print("Negative predictions:", negative)
    print("Missing predictions:", missing)
    print("Duplicate location-item-week rows:", duplicate_keys)
    print("Methods:")
    print(
        forecast.groupby("Forecast Method")
        .size()
        .div(4)
        .astype(int)
        .rename("series")
        .to_string()
    )

    last_data_week = pd.to_datetime(original["Week"]).max()
    recent_start = last_data_week - pd.Timedelta(days=28)
    series_last = forecast.groupby(["Fulfilment Location", item_col])["Last Observed Week"].max()
    on_final_week = int((series_last == last_data_week).sum())
    recent_earlier = int(
        ((series_last >= recent_start) & (series_last < last_data_week)).sum()
    )
    stale = int((series_last < recent_start).sum())
    print("Observed on the final week:", on_final_week)
    print("Recent, but not on the final week:", recent_earlier)
    print("Stale histories:", stale)

    problems = []
    if weeks != expected_weeks:
        problems.append(f"weeks are {weeks}")
    if not (combo_counts == 4).all():
        problems.append("a series does not have exactly 4 weeks")
    if original_combos != forecast_combos:
        problems.append("combo coverage mismatch")
    if negative or missing or duplicate_keys:
        problems.append("found negative, missing, or duplicate predictions")
    methods = set(forecast["Forecast Method"])
    if methods - {"xgboost", "recent_average_fallback"}:
        problems.append(f"unexpected methods {methods}")
    modelled = forecast["Forecast Method"].eq("xgboost")
    fallback = forecast["Forecast Method"].eq("recent_average_fallback")
    if modelled.any() and (forecast.loc[modelled, "History Weeks"] < MIN_HISTORY_WEEKS).any():
        problems.append("xgboost used on a short series")
    if fallback.any() and (forecast.loc[fallback, "History Weeks"] >= MIN_HISTORY_WEEKS).any():
        problems.append("fallback used on a long series")
    if fallback.any():
        flat = forecast.loc[fallback].groupby(["Fulfilment Location", item_col])[
            "Forecast Net Sold Quantity"
        ].nunique()
        if (flat != 1).any():
            problems.append("fallback forecast is not flat across the 4 weeks")
    if on_final_week + recent_earlier + stale != len(series_last):
        problems.append("freshness counts do not add up")
    if problems:
        raise AssertionError(f"{name} failed validation: {'; '.join(problems)}")
    print("Validation passed.")

print("\nAll four XGBoost forecast files passed validation.")


## 13. Combined model comparison

`model_comparison.csv` places XGBoost, Random Forest, and repeat-last-week side by side. It uses the evaluation table from this notebook. The lowest-MAE and lowest-median columns name every method that shares the best rounded value, so a tie stays a tie.

The one-step Random Forest numbers in `ml_model_evaluation.csv` answer a different question: each test week there still uses the real previous week. They are not copied into this comparison.


In [ ]:
comparison_columns = [
    "Dataset",
    "Horizon",
    "Cutoff start",
    "Cutoff end",
    "Series-origin pairs",
    "Scored weeks",
    "XGBoost MAE",
    "Random Forest MAE",
    "Repeat last week MAE",
    "XGBoost median AE",
    "Random Forest median AE",
    "Repeat last week median AE",
    "Lowest MAE",
    "Lowest median AE",
]
comparison_df = evaluation_df[comparison_columns].copy()
save_output("model_comparison.csv", comparison_df)
print(comparison_df.to_string(index=False))


## 14. How to read the scores

Read one dataset at a time, and read MAE and median AE separately.

- Lowest MAE means the smallest average absolute error across the scored weeks.
- Lowest median AE means the smallest error on a typical scored week.
- A few warehouse weeks of several thousand units can decide MAE while leaving the typical week almost unchanged.
- Levi's category has 15 series-cutoff pairs. That is a small sample.
- Series shorter than 20 weeks, and long series that do not have the cutoff week plus the next four observed weeks, are excluded from this score. They are still included in the March forecast files when they exist in the processed data. Short series use the recent-average fallback there.

The next cell states only what the saved table shows.


In [ ]:
print("Measured comparison")
print(
    "Random Forest was not retrained. Its errors come from "
    "outputs/ml_rolling_origin_evaluation.csv."
)
print(
    "Repeat-last-week was recomputed on the same series and cutoffs, "
    "and it matched that file before XGBoost was compared with it."
)

for name in datasets:
    part = comparison_df[comparison_df["Dataset"] == name]
    overall = part[part["Horizon"] == "All 4 weeks"].iloc[0]
    counts = evaluation_details[name]
    print(f"\n{name}")
    print(
        f"Series in the data: {counts['total_series']}. "
        f"Scored: {counts['series_scored']}. "
        f"Excluded: {counts['short_series'] + counts['long_unscored']} "
        f"({counts['short_series']} shorter than 20 weeks, "
        f"{counts['long_unscored']} long but missing a cutoff or a future week)."
    )
    print(
        "All 4 weeks MAE: "
        f"XGBoost {overall['XGBoost MAE']:.2f}, "
        f"Random Forest {overall['Random Forest MAE']:.2f}, "
        f"repeat last week {overall['Repeat last week MAE']:.2f}. "
        f"Lowest MAE: {overall['Lowest MAE']}."
    )
    print(
        "All 4 weeks median AE: "
        f"XGBoost {overall['XGBoost median AE']:.2f}, "
        f"Random Forest {overall['Random Forest median AE']:.2f}, "
        f"repeat last week {overall['Repeat last week median AE']:.2f}. "
        f"Lowest median AE: {overall['Lowest median AE']}."
    )
    for _, row in part[part["Horizon"] != "All 4 weeks"].iterrows():
        print(
            f"  {row['Horizon']}: lowest MAE is {row['Lowest MAE']}; "
            f"lowest median AE is {row['Lowest median AE']}."
        )

print("\nRuntime seconds")
for label, seconds in timings.items():
    print(f"{label}: {seconds}")


## 15. Measured results from this run

The cell above prints these figures from the tables just saved. This is the reading of that run.

XGBoost is not the best method across the four datasets.

- Levi's SKU, 896 scored weeks: repeat-last-week has the lowest MAE (4.08) and the lowest median AE (0.00). XGBoost MAE is 6.93 and Random Forest MAE is 7.24.
- Souled Store SKU, 5,844 scored weeks: Random Forest has the lowest MAE (5.59) and the lowest median AE (1.90). XGBoost MAE is 6.23 and repeat-last-week MAE is 5.97.
- Levi's category, 60 scored weeks from 2 series and 15 series-cutoff pairs: XGBoost has a slightly lower MAE (1,263.64) than Random Forest (1,270.06). Repeat-last-week MAE is 1,742.63. The typical week is closer under repeat-last-week (median AE 44.00, versus 126.32 for XGBoost).
- Souled Store category, 2,112 scored weeks: Random Forest has the lowest MAE (12.63). XGBoost MAE is 13.94 and repeat-last-week MAE is 13.43. Median AE is 2.00 for Random Forest and repeat-last-week, and 2.32 for XGBoost.

On individual weeks, repeat-last-week is closer than XGBoost more often in every dataset. The full table, the exclusion counts, and the limits of this comparison are written up in `docs/results_and_analysis.md`.
